# Lab 31 (solution): Corrective RAG (CRAG) from scratch

Reference implementation. Builds on Lab 06's corpus, chunker, and numpy index. The new material is the **retrieval evaluator**, **knowledge refinement** (decompose-then-recompose), and the **corrective actions** (rewrite-and-retry, web-search fallback).

Pattern source: Yan et al. 2024, *Corrective Retrieval Augmented Generation* ([arXiv:2401.15884](https://arxiv.org/abs/2401.15884)).

## Step 0: Setup

Same provider-agnostic pattern as prior labs. No new dependencies beyond Lab 06 (`sentence-transformers`, `numpy`).

In [ ]:
import hashlib
import json
import os
import pathlib
import re
from typing import Any
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")

In [ ]:
def chat(messages: list[dict], temperature: float = 0.0) -> str:
    """Provider-agnostic plain chat. Returns assistant text."""
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, temperature=temperature)
        return resp.choices[0].message.content or ""
    else:
        from anthropic import Anthropic
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        resp = Anthropic().messages.create(
            model=MODEL, system=system, messages=non_system,
            max_tokens=1024, temperature=temperature)
        return "".join(b.text for b in resp.content if hasattr(b, "text"))


def chat_json(messages: list[dict]) -> dict:
    """Chat call expected to return a JSON object. Strips code fences, parses."""
    raw = chat(messages).strip()
    raw = re.sub(r"^```(json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # last-ditch: grab the outermost {...}
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return json.loads(m.group(0)) if m else {"_parse_error": raw[:200]}

## Step 1: Reuse Lab 06's retrieval stack

We point at Lab 06's corpus and rebuild the index. Nothing new here — this is the baseline CRAG corrects.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Reuse Lab 06's corpus — no need to duplicate it.
CORPUS_DIR = pathlib.Path("../06-agentic-rag-from-scratch/corpus")
TARGET_TOKENS, OVERLAP_TOKENS = 160, 32

def approx_tokens(t): return int(len(t.split()) / 0.75)
def split_paras(t): return [p.strip() for p in re.split(r"\n\s*\n", t) if p.strip()]
def split_sents(t): return [p.strip() for p in re.split(r"(?<=[.!?])\s+", t) if p.strip()]

def chunk_text(text, target=TARGET_TOKENS):
    out, cur, cur_tok = [], [], 0
    for para in split_paras(text):
        pt = approx_tokens(para)
        if cur_tok + pt > target and cur:
            out.append("\n\n".join(cur))
            cur, cur_tok = [], 0
        if pt > target:
            for sent in split_sents(para):
                st = approx_tokens(sent)
                if cur_tok + st > target and cur:
                    out.append(" ".join(cur))
                    cur, cur_tok = [], 0
                cur.append(sent)
                cur_tok += st
        else:
            cur.append(para)
            cur_tok += pt
    if cur:
        out.append("\n\n".join(cur))
    return out

all_chunks = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    doc_id = path.stem
    body = path.read_text()
    title = body.splitlines()[0].lstrip("# ").strip() if body else doc_id
    for i, ch in enumerate(chunk_text(body)):
        all_chunks.append({"doc_id": doc_id, "chunk_id": f"{doc_id}#{i}",
                           "title": title, "text": ch})
print(f"Loaded {len({c['doc_id'] for c in all_chunks})} docs, "
      f"{len(all_chunks)} chunks")

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
embeddings = embedder.encode([c["text"] for c in all_chunks],
                             normalize_embeddings=True, convert_to_numpy=True,
                             show_progress_bar=False)
print(f"Index ready: {embeddings.shape}")

In [ ]:
def search_corpus(query: str, top_k: int = 5) -> list[dict]:
    """Return top-k chunks with cosine scores. (Lab 06's retriever, condensed.)"""
    q = embedder.encode([query], normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=False)[0]
    scores = embeddings @ q
    idx = np.argsort(scores)[::-1][:top_k]
    return [{**all_chunks[i], "score": float(scores[i])} for i in idx]

# Smoke test
for r in search_corpus("how does the ReAct loop work?", top_k=3):
    print(f"  {r['score']:.3f}  {r['chunk_id']}  {r['text'][:60]}...")

## Step 2: The retrieval evaluator

The one extra model call that makes CRAG corrective. It grades each retrieved passage and returns an overall verdict in `{correct, ambiguous, incorrect}`.

In [ ]:
def grade_retrieval(query: str, chunks: list[dict]) -> dict:
    """CRAG retrieval evaluator. Score each chunk\'s relevance to the query and
    return an overall verdict in {correct, ambiguous, incorrect}.

    This is the one extra model call that distinguishes CRAG from static RAG.
    We ask for a per-chunk score and an aggregate so we can both route and
    refine. A fine-tuned lightweight evaluator (as in the paper) would replace
    this LLM call in production."""
    numbered = "\n".join(f"[{i}] {c['text']}" for i, c in enumerate(chunks))
    prompt = [
        {"role": "system", "content":
         "You grade whether retrieved passages contain the information needed "
         "to answer a query. Reply with JSON only."},
        {"role": "user", "content":
         f"Query: {query}\n\nPassages:\n{numbered}\n\n"
         "Return JSON: {\"scores\": [float per passage 0..1], "
         "\"verdict\": \"correct|ambiguous|incorrect\", \"reason\": str}. "
         "correct = passages fully answer it; incorrect = unrelated/missing the "
         "key fact; ambiguous = partial."},
    ]
    out = chat_json(prompt)
    out.setdefault("scores", [0.0] * len(chunks))
    out.setdefault("verdict", "ambiguous")
    return out

## Step 3: Knowledge refinement (decompose-then-recompose)

Strip retrieved chunks to the sentences that carry signal, keeping chunk-id provenance so citations survive. Don't hand the generator a wall of mostly-irrelevant text.

In [ ]:
def decompose_recompose(query: str, chunks: list[dict],
                        scores: list[float], keep_threshold: float = 0.5) -> str:
    """CRAG knowledge refinement: split retrieved chunks into knowledge strips,
    keep only the strips relevant to the query, recompose. We approximate the
    paper\'s strip-level filtering with sentence-level filtering scored by the
    grader\'s per-chunk scores plus a cheap relevance pass.

    The point: don\'t hand the generator a wall of mostly-irrelevant context.
    Strip it to the load-bearing sentences first."""
    kept = []
    for c, s in zip(chunks, scores, strict=False):
        if s < keep_threshold:
            continue
        for sent in split_sents(c["text"]):
            kept.append({"chunk_id": c["chunk_id"], "sentence": sent})
    if not kept:
        return ""
    # Recompose with provenance so citations survive refinement.
    return "\n".join(f"[{k['chunk_id']}] {k['sentence']}" for k in kept)

## Step 4: Corrective actions

For `incorrect`, fall back to web search (stubbed here so the lab runs offline). For `ambiguous`, rewrite the query and retry, then combine.

In [ ]:
def web_search_fallback(query: str) -> str:
    """Stand-in for a real web-search tool. In production, call a search API and
    return formatted snippets. Here we return a sentinel so the loop is runnable
    offline and you can see the routing decision without a search dependency."""
    return f"[web-fallback] (replace with real search results for: {query})"


def rewrite_query(query: str) -> str:
    """Ambiguous-retrieval corrective action: reformulate for a second attempt."""
    out = chat([
        {"role": "system", "content": "Rewrite the query to be more retrievable. "
         "Reply with only the rewritten query."},
        {"role": "user", "content": query},
    ])
    return out.strip() or query

## Step 5: The CRAG loop

Wire it together: retrieve, grade, route, generate. The trace records each routing decision so you can debug.

In [ ]:
def corrective_rag(query: str, top_k: int = 5, verbose: bool = True) -> dict:
    """Full CRAG loop: retrieve -> grade -> {refine | rewrite+retry | fallback}
    -> generate. Returns the answer plus a trace of the routing decisions."""
    trace = []
    chunks = search_corpus(query, top_k=top_k)
    grade = grade_retrieval(query, chunks)
    verdict = grade["verdict"]
    trace.append(("grade", verdict, grade.get("reason", "")))
    if verbose:
        print(f"  verdict={verdict}: {grade.get('reason','')[:80]}")

    if verdict == "correct":
        context = decompose_recompose(query, chunks, grade["scores"])
    elif verdict == "incorrect":
        context = web_search_fallback(query)
        trace.append(("action", "web_fallback", ""))
        if verbose:
            print("  -> discarded retrieval, used web fallback")
    else:  # ambiguous: rewrite + retry once, then combine
        rq = rewrite_query(query)
        trace.append(("action", "rewrite", rq))
        chunks2 = search_corpus(rq, top_k=top_k)
        grade2 = grade_retrieval(rq, chunks2)
        refined = decompose_recompose(rq, chunks2, grade2["scores"])
        context = (refined + "\n" + web_search_fallback(query)).strip()
        if verbose:
            print(f"  -> rewrote to '{rq[:50]}', combined with fallback")

    answer = chat([
        {"role": "system", "content": "Answer using only the evidence. Cite chunk "
         "ids as [doc#n]. If evidence is insufficient, say so explicitly."},
        {"role": "user", "content": f"Evidence:\n{context}\n\nQuestion: {query}"},
    ])
    return {"query": query, "verdict": verdict, "answer": answer, "trace": trace}

## Step 6: See it correct a retrieval failure

The payoff: on an off-corpus query, static RAG grounds an answer in whatever it retrieved and tends to fabricate; CRAG's grader catches the gap and routes away.

In [ ]:
# Demonstration: a query the corpus CAN answer vs one it CANNOT.
print("=== Query 1: in-corpus (expect verdict=correct) ===")
r1 = corrective_rag("What is the ReAct pattern?")
print(r1["answer"][:300], "\n")

print("=== Query 2: off-corpus (expect verdict=incorrect -> fallback) ===")
r2 = corrective_rag("What were Q3 2025 earnings for Acme Corp?")
print(r2["answer"][:300])
# Static RAG would ground an answer in whatever it retrieved and likely
# fabricate. CRAG\'s grader catches that the corpus does not contain the
# answer and routes to the fallback (or abstention) instead.

## Step 7: Calibrate the grader

The grader is itself a model call and can be wrong in both directions. Measure its accuracy as its own number.

In [ ]:
# Calibrating the grader is the whole game. Measure its verdict distribution
# on a small labeled set; a grader that returns "correct" 95% of the time is not
# buying you anything, and one that is too strict pays for needless fallbacks.
labeled = [
    ("What is the agent loop?", "correct"),
    ("How does chunking work?", "correct"),
    ("What is the capital of France?", "incorrect"),   # off-corpus
    ("Tell me about vector indexes and also the weather", "ambiguous"),
]
hits = 0
for q, expected in labeled:
    chunks = search_corpus(q, top_k=5)
    got = grade_retrieval(q, chunks)["verdict"]
    ok = got == expected
    hits += ok
    print(f"  {'OK ' if ok else 'XX '} q={q[:40]:42} expected={expected:10} got={got}")
print(f"\nGrader accuracy on this tiny set: {hits}/{len(labeled)}")
print("In a real lab you would expand this to your eval_set.jsonl and track it in CI.")

## What you built

A from-scratch CRAG loop: a retrieval evaluator, sentence-level knowledge refinement with provenance, and two corrective actions (rewrite-and-retry, fallback). The cost over static RAG is one grader call per query — among the cheapest SOTA patterns to adopt.

**Where this implementation simplifies:** the grader is an LLM call, not the paper's fine-tuned lightweight evaluator; the web fallback is stubbed; refinement is sentence-level, not the paper's strip-level. Each is marked in the code and is a reasonable next extension.

See [`concepts/rag/sota-rag-patterns.md`](../../../concepts/rag/sota-rag-patterns.md) and [`recipes/rag/03-corrective-rag.md`](../../../recipes/rag/03-corrective-rag.md).